# Crear train balanceado por celda de Voronoi

Genera `train_celdas_balanceado.parquet`: un train con la **misma cantidad de imágenes
por celda** (cluster). Las celdas y sus prototipos salen de `assets/prototipos.csv`
(`ind_cluster, lat, lon`); `N_CLUSTERS` es la cantidad de filas de ese CSV y cada celda recibe
`TOTAL / N_CLUSTERS` imágenes (si no divide exacto, las primeras `TOTAL % N_CLUSTERS` celdas,
por `ind_cluster`, reciben una más).

- **Asignación a celdas**: cada imagen va al prototipo más cercano por distancia angular
  (mayor producto punto entre vectores unitarios), igual que en `celdas_voronoi.ipynb`.
- **Fuente**: todo el dataset de `DATA_URL` (`train-*.parquet`), no una muestra previa.
- **Sin fuga con la validación global**: se excluyen las filas cuyo `panoid` **o** `image_id`
  figura en `datasets/samples/data/validation/validation.parquet`. Ese archivo no se modifica
  y solo se leen sus IDs y coordenadas, sin cargar imágenes.
- **Un panorama, una imagen** (`UNA_IMAGEN_POR_PANORAMA=True`): cada `panoid` aporta a lo sumo
  una vista, elegida al azar. Evita que varias vistas casi idénticas ocupen el cupo de una celda.
- **Celdas chicas**: si una celda tiene menos filas disponibles que su cupo, se toman todas y se
  informa el déficit; el total final puede quedar por debajo de `TOTAL`.
- La selección es sin reemplazo y reproducible con `SEMILLA`. Las imágenes se copian sin decodificar.

Salidas en `CARPETA_SALIDA`: `train_celdas_balanceado.parquet` (todas las columnas más
`ind_cluster`), `resumen_celdas.csv` y `config_celdas_balanceado.json`.

In [1]:
from pathlib import Path
import json
import hashlib
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from tqdm.auto import tqdm

# Parámetros editables.
TOTAL = 20_000  # Total base; el cupo de una celda es MULT_CONT[continente] * TOTAL / N_CLUSTERS (ver celda de multiplicadores).
ARCHIVO_PROTOTIPOS = "src/notebooks/preprocessing/assets/prototipos.csv"  # ind_cluster, lat, lon
VALIDACION_GLOBAL = "datasets/samples/data/validation/validation.parquet"
UNA_IMAGEN_POR_PANORAMA = True  # True: cada panoid aporta a lo sumo una imagen.
SEMILLA = 42
PATRON_ARCHIVOS = "**/train-*.parquet"
CARPETA_SALIDA = "datasets/samples/data/train_aumentado_NA_EU"  # Relativa a la raíz del proyecto, o absoluta.
COL_LAT, COL_LON = "latitude", "longitude"
TAMANO_LOTE = 128  # Limita la memoria al leer imágenes.

## Localizar el dataset y los prototipos

`DATA_URL` se toma del entorno si ya está definida; de lo contrario se lee del `.env`. Las rutas
relativas se resuelven respecto de la carpeta del `.env`. El CSV debe tener `ind_cluster` consecutivo
desde 0 y coordenadas válidas; su huella SHA-256 queda en la configuración de salida.

In [2]:
def entero_valido(valor, nombre, minimo):
    if isinstance(valor, bool) or not isinstance(valor, (int, np.integer)) or valor < minimo:
        raise ValueError(f"{nombre} debe ser un entero >= {minimo}.")

for nombre in ("TOTAL", "TAMANO_LOTE"):
    entero_valido(globals()[nombre], nombre, 1)
entero_valido(SEMILLA, "SEMILLA", 0)
if not isinstance(UNA_IMAGEN_POR_PANORAMA, bool):
    raise ValueError("UNA_IMAGEN_POR_PANORAMA debe ser True o False.")

env_path = next(
    (carpeta / ".env" for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No se encontró el .env del proyecto desde el directorio actual.")

valor_env = ""
for linea in env_path.read_text(encoding="utf-8-sig").splitlines():
    clave, separador, valor = linea.partition("=")
    if separador and clave.strip() == "DATA_URL":
        valor_env = valor.strip().strip("\"'")

ruta_configurada = os.environ.get("DATA_URL", valor_env).strip()
if not ruta_configurada:
    raise ValueError("Configurá DATA_URL en el .env con la carpeta del dataset local.")

raiz_proyecto = env_path.parent


def resolver(ruta):
    ruta = Path(ruta).expanduser()
    return (ruta if ruta.is_absolute() else raiz_proyecto / ruta).resolve()


ruta_datos = resolver(ruta_configurada)
if not ruta_datos.is_dir():
    raise NotADirectoryError(f"DATA_URL apunta a una carpeta inexistente: {ruta_datos}")
archivos = sorted(p for p in ruta_datos.glob(PATRON_ARCHIVOS) if p.is_file())
if not archivos:
    raise FileNotFoundError(f"No se encontraron Parquet con {PATRON_ARCHIVOS!r} en {ruta_datos}")

raiz_salida = resolver(CARPETA_SALIDA)
if raiz_salida == ruta_datos or ruta_datos in raiz_salida.parents:
    raise ValueError("CARPETA_SALIDA debe estar fuera de la carpeta del dataset de entrada.")
ruta_train = raiz_salida / "train_celdas_balanceado.parquet"
ruta_resumen = raiz_salida / "resumen_celdas.csv"
ruta_config = raiz_salida / "config_celdas_balanceado.json"
existentes = [p for p in (ruta_train, ruta_resumen, ruta_config) if p.exists()]
if existentes:
    raise FileExistsError(f"La salida ya existe: {existentes}. No se sobrescribe.")

print(f"Dataset: {ruta_datos}")
print(f"Archivos de entrada: {len(archivos):,}")


def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])


def asignar(X, C, chunk=200_000):
    # Celda del prototipo más cercano: mayor producto punto = menor distancia angular.
    labels = np.empty(len(X), dtype=np.int32)
    for i in range(0, len(X), chunk):
        labels[i:i + chunk] = (X[i:i + chunk] @ C.T).argmax(axis=1)
    return labels


def coordenadas_validas(lat, lon):
    return np.isfinite(lat) & np.isfinite(lon) & (np.abs(lat) <= 90) & (np.abs(lon) <= 180)


ruta_prototipos = resolver(ARCHIVO_PROTOTIPOS)
prototipos = pd.read_csv(ruta_prototipos)
if not {"ind_cluster", "lat", "lon"}.issubset(prototipos.columns):
    raise ValueError("El CSV de prototipos debe tener ind_cluster, lat y lon.")
prototipos = prototipos.sort_values("ind_cluster").reset_index(drop=True)
N_CLUSTERS = len(prototipos)
if N_CLUSTERS == 0 or not (prototipos["ind_cluster"].to_numpy() == np.arange(N_CLUSTERS)).all():
    raise ValueError("ind_cluster debe ser 0..N-1 sin repetidos ni huecos.")
if not coordenadas_validas(prototipos["lat"].to_numpy(float), prototipos["lon"].to_numpy(float)).all():
    raise ValueError("Hay prototipos con coordenadas inválidas.")
if TOTAL < N_CLUSTERS:
    raise ValueError(f"TOTAL ({TOTAL:,}) debe ser >= N_CLUSTERS ({N_CLUSTERS}).")
centroides = to_xyz(prototipos["lat"].to_numpy(float), prototipos["lon"].to_numpy(float))
huella_prototipos = hashlib.sha256(ruta_prototipos.read_bytes()).hexdigest()
print(f"Prototipos: {N_CLUSTERS}; imágenes por celda: {TOTAL / N_CLUSTERS:.2f}")

Dataset: D:\world-streetview-500k\parquet\data
Archivos de entrada: 393
Prototipos: 70; imágenes por celda: 285.71


## Multiplicador de cupo por continente

Cada celda se asigna a un continente según las coordenadas de su prototipo (reglas aproximadas por
latitud/longitud; revisá la tabla impresa y corregí casos dudosos con `CONTINENTE_MANUAL`). El cupo de
una celda es `MULT_CONT[continente] * TOTAL / N_CLUSTERS`, redondeado con el método del resto mayor.
`TOTAL` pasa a ser el total **base**; el total oficial, `TOTAL_OFICIAL`, es la suma de los cupos y vale
`TOTAL * promedio(MULT_CONT por celda)`. Con todos los multiplicadores en 1.0 el resultado es igual al anterior.

In [8]:
# Multiplicador de cupo por continente: cupo de una celda = MULT_CONT[continente] * TOTAL / N_CLUSTERS.
MULT_CONT = {
    "Africa": 1.0,
    "America del Norte": 3.0,
    "America del Sur": 1.0,
    "Asia": 1.0,
    "Europa": 2.0,
    "Oceania": 1.0,
    "Antartida": 1.0,
}
CONTINENTE_MANUAL = {
    12: "America del Sur",
    51: "America del Sur"
}  # Correcciones: {ind_cluster: "Continente"}, p. ej. {12: "America del Norte"}.


def continente(lat, lon):
    # Reglas aproximadas por caja de coordenadas; alcanzan para clasificar centros de celdas.
    if lat < -60:
        return "Antartida"
    if (lat < -10 and lon > 110) or (lat < 2 and lon > 141) or (lon < -140 and lat < 30):
        return "Oceania"
    if lon < -30:
        if lat >= 12 or (lat >= 7 and lon < -77.5):
            return "America del Norte"
        return "America del Sur"
    if lat > 12 and lon > 38.5 or lat > 27 and lon > 33.5:
        return "Asia"
    if lat < 36 and -20 <= lon <= 52:
        return "Africa"
    if (lat >= 36 and lon <= 26) or (lat >= 44 and lon <= 60) or (lat >= 41 and lon <= 40) or lon < -10:
        return "Europa"
    return "Asia"


continente_celda = pd.Series(
    [continente(la, lo) for la, lo in zip(prototipos["lat"], prototipos["lon"])], index=range(N_CLUSTERS),
)
for celda, nombre in CONTINENTE_MANUAL.items():
    if celda not in continente_celda.index:
        raise ValueError(f"CONTINENTE_MANUAL: ind_cluster {celda} no existe.")
    continente_celda[celda] = nombre
desconocidos = set(continente_celda) - set(MULT_CONT)
if desconocidos:
    raise ValueError(f"Continentes sin multiplicador en MULT_CONT: {sorted(desconocidos)}")
if any(not isinstance(m, (int, float)) or isinstance(m, bool) or not m > 0 for m in MULT_CONT.values()):
    raise ValueError("Los multiplicadores de MULT_CONT deben ser números > 0.")

mult_celda = continente_celda.map(MULT_CONT).astype(float)
objetivo_celda = mult_celda * TOTAL / N_CLUSTERS
cupos_celda = np.floor(objetivo_celda).astype("int64")
faltan = int(round(objetivo_celda.sum())) - int(cupos_celda.sum())
extra = (objetivo_celda - cupos_celda).sort_values(ascending=False, kind="stable").index[:faltan]
cupos_celda.loc[extra] += 1  # Resto mayor: el total es exactamente round(sum(objetivo)).
TOTAL_OFICIAL = int(cupos_celda.sum())
if cupos_celda.min() < 1:
    raise ValueError("Algún multiplicador deja una celda sin cupo; subí TOTAL o el multiplicador.")

tabla_continentes = pd.DataFrame({
    "celdas": continente_celda.value_counts(),
    "mult": pd.Series(MULT_CONT),
    "cupo_por_celda": cupos_celda.groupby(continente_celda).mean().round(1),
    "cupo_total": cupos_celda.groupby(continente_celda).sum(),
}).dropna(subset=["celdas"])
display(tabla_continentes)
print(f"TOTAL base: {TOTAL:,}; TOTAL_OFICIAL (suma de cupos): {TOTAL_OFICIAL:,}")

,celdas,mult,cupo_por_celda,cupo_total
Africa,8.0,1.0,285.8,2286.0
America del Norte,7.0,3.0,857.0,5999.0
America del Sur,13.0,1.0,285.8,3715.0
Asia,22.0,1.0,286.0,6292.0
Europa,15.0,2.0,571.0,8565.0
Oceania,5.0,1.0,285.8,1429.0


TOTAL base: 20,000; TOTAL_OFICIAL (suma de cupos): 28,286


In [7]:
import plotly.graph_objects as go

# Prototipos coloreados por continente; sirve para revisar CONTINENTE_MANUAL antes de generar el train.
colores_continente = {
    "Africa": "#e69f00", "America del Norte": "#0072b2", "America del Sur": "#009e73",
    "Asia": "#d55e00", "Europa": "#cc79a7", "Oceania": "#56b4e9", "Antartida": "#666666",
}
fig_cont = go.Figure()
for nombre in colores_continente:
    celdas_cont = continente_celda.index[continente_celda == nombre]
    if len(celdas_cont) == 0:
        continue
    fig_cont.add_trace(go.Scattergeo(
        lat=prototipos.loc[celdas_cont, "lat"], lon=prototipos.loc[celdas_cont, "lon"],
        mode="markers+text", text=[str(c) for c in celdas_cont], textposition="top center",
        hovertext=[f"celda {c}: {nombre}, x{MULT_CONT[nombre]:g}, cupo {cupos_celda[c]}" for c in celdas_cont],
        hoverinfo="text", name=f"{nombre} ({len(celdas_cont)})",
        marker=dict(size=10, color=colores_continente[nombre], line=dict(width=1, color="black")),
    ))
fig_cont.update_geos(projection_type="natural earth", showland=True, landcolor="#f1ede4", showocean=True,
                     oceancolor="#dfeaf2", showcountries=True, countrycolor="#bbbbbb", showcoastlines=True,
                     coastlinecolor="#888888")
fig_cont.update_layout(title="Prototipos por continente (número = ind_cluster)", height=650,
                       margin=dict(l=0, r=0, t=50, b=0), legend=dict(orientation="h", y=-0.02))
fig_cont.show()

## Leer la validación global

Se leen `image_id`, `panoid` y las coordenadas (sin imágenes en memoria). Los IDs se usan para
excluir de train; las coordenadas, solo para informar cuántas imágenes de validación cae en cada celda.

In [9]:
ruta_validacion = resolver(VALIDACION_GLOBAL)
if not ruta_validacion.is_file():
    raise FileNotFoundError(f"Falta la validación global: {ruta_validacion}")
estado_validacion = ruta_validacion.stat()
columnas_val = ["image_id", "panoid", COL_LAT, COL_LON]
with pq.ParquetFile(ruta_validacion) as parquet:
    if not set(columnas_val).issubset(parquet.schema_arrow.names):
        raise ValueError(f"La validación global debe tener {columnas_val}.")
    validacion = parquet.read(columns=columnas_val).to_pandas()
for columna in ("image_id", "panoid"):
    validacion[columna] = validacion[columna].astype("string").str.strip().replace("", pd.NA)
if validacion.empty or validacion[["image_id", "panoid"]].isna().any().any():
    raise ValueError("La validación global debe tener filas e identificadores no vacíos.")
if validacion["image_id"].duplicated().any():
    raise ValueError("La validación global contiene image_id duplicados.")

ids_ordenados = validacion[["image_id", "panoid"]].sort_values(["image_id", "panoid"]).to_numpy().tolist()
huella_ids = hashlib.sha256(json.dumps(ids_ordenados, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()
info_validacion = {
    "ruta": str(ruta_validacion), "filas": len(validacion),
    "panoramas": int(validacion["panoid"].nunique()), "columnas_exclusion": ["panoid", "image_id"],
    "sha256_ids": huella_ids, "bytes": estado_validacion.st_size, "mtime_ns": estado_validacion.st_mtime_ns,
}
lat_val = pd.to_numeric(validacion[COL_LAT], errors="coerce").to_numpy(float)
lon_val = pd.to_numeric(validacion[COL_LON], errors="coerce").to_numpy(float)
ok_val = coordenadas_validas(lat_val, lon_val)
celdas_val = asignar(to_xyz(lat_val[ok_val], lon_val[ok_val]), centroides)
validacion_por_celda = pd.Series(np.bincount(celdas_val, minlength=N_CLUSTERS), index=range(N_CLUSTERS))
print(f"Validación global: {len(validacion):,} imágenes; {info_validacion['panoramas']:,} panoramas reservados.")
print(f"Celdas sin imágenes de validación: {(validacion_por_celda == 0).sum()} de {N_CLUSTERS}")

Validación global: 5,332 imágenes; 1,333 panoramas reservados.
Celdas sin imágenes de validación: 0 de 70


## Asignar celdas y calcular la disponibilidad

La primera pasada lee solo coordenadas e IDs de todos los archivos y asigna cada fila a su celda.
Se mantiene el índice global sobre los archivos ordenados para copiar después las filas elegidas.
Quedan fuera las filas con IDs vacíos o coordenadas inválidas y las reservadas por la validación
global (por `panoid` o `image_id`).

In [10]:
partes_celda, partes_image, partes_pano = [], [], []
inventario = []
esquema = None
columnas_lectura = [COL_LAT, COL_LON, "image_id", "panoid"]
for archivo in tqdm(archivos, desc="Asignando celdas"):
    with pq.ParquetFile(archivo) as parquet:
        if not set(columnas_lectura).issubset(parquet.schema_arrow.names):
            raise ValueError(f"Faltan columnas {columnas_lectura} en {archivo}")
        if esquema is None:
            esquema = parquet.schema_arrow
        elif not esquema.equals(parquet.schema_arrow, check_metadata=True):
            raise ValueError(f"El esquema difiere del primer archivo: {archivo}")
        tabla = parquet.read(columns=columnas_lectura).to_pandas()
        inventario.append({
            "archivo": str(archivo), "filas": parquet.metadata.num_rows,
            "bytes": archivo.stat().st_size, "mtime_ns": archivo.stat().st_mtime_ns,
        })
    lat = pd.to_numeric(tabla[COL_LAT], errors="coerce").to_numpy(float)
    lon = pd.to_numeric(tabla[COL_LON], errors="coerce").to_numpy(float)
    valida = coordenadas_validas(lat, lon)
    celda = np.full(len(tabla), -1, dtype=np.int16)
    celda[valida] = asignar(to_xyz(lat[valida], lon[valida]), centroides)
    partes_celda.append(celda)
    partes_image.append(tabla["image_id"].astype("string").str.strip().replace("", pd.NA))
    partes_pano.append(tabla["panoid"].astype("string").str.strip().replace("", pd.NA))
    del tabla

celdas = np.concatenate(partes_celda)
image_ids = pd.concat(partes_image, ignore_index=True)
panoids = pd.concat(partes_pano, ignore_index=True)
del partes_celda, partes_image, partes_pano

if image_ids.dropna().duplicated().any():
    raise ValueError("La fuente contiene image_id duplicados.")
ids_invalidos = (image_ids.isna() | panoids.isna()).to_numpy()
coords_invalidas = celdas < 0
reservadas = (image_ids.isin(validacion["image_id"]) | panoids.isin(validacion["panoid"])).to_numpy()
elegibles = ~ids_invalidos & ~coords_invalidas & ~reservadas
print(f"Filas totales: {len(celdas):,}")
print(f"IDs inválidos: {ids_invalidos.sum():,}; coordenadas inválidas: {coords_invalidas.sum():,}")
print(f"Excluidas por validación global (panoid o image_id): {(reservadas & ~ids_invalidos & ~coords_invalidas).sum():,}")
print(f"Elegibles para train: {elegibles.sum():,}")

Asignando celdas:   0%|          | 0/393 [00:00<?, ?it/s]

Filas totales: 1,999,060
IDs inválidos: 0; coordenadas inválidas: 0
Excluidas por validación global (panoid o image_id): 5,332
Elegibles para train: 1,993,728


## Calcular cupos y seleccionar

Se mezclan al azar (con `SEMILLA`) las filas elegibles y, si `UNA_IMAGEN_POR_PANORAMA`, se conserva la
primera fila de cada `panoid` (una vista al azar por panorama, de forma global: ningún panorama queda
en dos celdas). Luego cada celda toma el prefijo de su cupo. Si la disponibilidad no alcanza, se toma todo
y se informa el déficit.

In [11]:
rng = np.random.default_rng(SEMILLA)
indices_elegibles = np.flatnonzero(elegibles)
candidatos = pd.DataFrame({
    "indice": indices_elegibles,
    "celda": celdas[indices_elegibles].astype(np.int64),
    "panoid": panoids.to_numpy()[indices_elegibles],
}).iloc[rng.permutation(len(indices_elegibles))]
filas_elegibles = candidatos.groupby("celda").size().reindex(range(N_CLUSTERS), fill_value=0)
if UNA_IMAGEN_POR_PANORAMA:
    candidatos = candidatos.drop_duplicates("panoid", keep="first")
disponibles = candidatos.groupby("celda").size().reindex(range(N_CLUSTERS), fill_value=0)

cupos = cupos_celda.copy()  # MULT_CONT[continente] * TOTAL / N_CLUSTERS por celda.
seleccionadas = np.minimum(cupos, disponibles).astype("int64")
deficit = cupos - seleccionadas

indices_train = np.sort(np.concatenate([
    grupo["indice"].to_numpy()[:int(seleccionadas[celda])]
    for celda, grupo in candidatos.groupby("celda", sort=True)
])).astype(np.int64)
del candidatos

assert len(indices_train) == seleccionadas.sum()
assert len(np.unique(indices_train)) == len(indices_train)
assert elegibles[indices_train].all()
assert np.array_equal(np.bincount(celdas[indices_train], minlength=N_CLUSTERS), seleccionadas.to_numpy())
if UNA_IMAGEN_POR_PANORAMA:
    assert not panoids.iloc[indices_train].duplicated().any()

resumen = pd.DataFrame({
    "lat": prototipos["lat"], "lon": prototipos["lon"],
    "continente": continente_celda, "mult": mult_celda,
    "filas_elegibles": filas_elegibles, "disponibles": disponibles,
    "cupo": cupos, "seleccionadas": seleccionadas, "deficit": deficit,
    "validacion": validacion_por_celda,
})
resumen.index.name = "ind_cluster"
display(resumen.drop(columns="continente").describe().round(1))
if deficit.any():
    print(f"AVISO: {(deficit > 0).sum()} celdas con menos filas que su cupo (faltan {deficit.sum():,}).")
    display(resumen[resumen["deficit"] > 0])
print(f"Cupo por celda: entre {cupos.min()} y {cupos.max()}; filas de salida: {len(indices_train):,} de {TOTAL_OFICIAL:,}")

,lat,lon,mult,filas_elegibles,disponibles,cupo,seleccionadas,deficit,validacion
count,70.0,70.0,70.0,70.0,70.0,70.0,70.0,70.0,70.0
mean,17.6,21.6,1.4,28481.8,7120.5,404.1,402.9,1.2,76.2
std,31.6,77.9,0.7,35951.8,8987.9,191.3,190.5,9.7,94.4
min,-44.2,-114.5,1.0,1140.0,285.0,285.0,285.0,0.0,4.0
25%,-4.8,-53.9,1.0,7721.0,1930.2,286.0,286.0,0.0,20.0
50%,20.9,21.4,1.0,17074.0,4268.5,286.0,286.0,0.0,48.0
75%,46.2,92.2,2.0,34584.0,8646.0,571.0,571.0,0.0,95.0
max,66.3,174.1,3.0,197192.0,49298.0,857.0,857.0,81.0,512.0


AVISO: 2 celdas con menos filas que su cupo (faltan 82).


,lat,lon,continente,mult,filas_elegibles,disponibles,cupo,seleccionadas,deficit,validacion
ind_cluster,,,,,,,,,,
12,10.370717,-84.989867,America del Sur,1.0,1140,285,286,285,1,16
26,65.094900,-19.496907,Europa,2.0,1960,490,571,490,81,20


Cupo por celda: entre 285 y 857; filas de salida: 28,204 de 28,286


## Guardar

Se escribe `train_celdas_balanceado.parquet` por lotes, en orden de origen, con todas las columnas y el
esquema original más `ind_cluster` (int32). Se escribe a un archivo temporal que se renombra al final.
Después se relee lo guardado para comprobar cupos por celda, ausencia de IDs reservados y unicidad.

In [12]:
def comprobar_estado():
    for archivo, info in zip(archivos, inventario, strict=True):
        estado = archivo.stat()
        if estado.st_size != info["bytes"] or estado.st_mtime_ns != info["mtime_ns"]:
            raise RuntimeError(f"La fuente cambió desde el conteo: {archivo}. Reejecutá desde el inicio.")
    estado = ruta_validacion.stat()
    if estado.st_size != info_validacion["bytes"] or estado.st_mtime_ns != info_validacion["mtime_ns"]:
        raise RuntimeError("La validación global cambió. Reejecutá desde el inicio.")


def guardar_train(destino, tamano_lote):
    comprobar_estado()
    destino.parent.mkdir(parents=True, exist_ok=True)
    esquema_salida = esquema.append(pa.field("ind_cluster", pa.int32()))
    destino_tmp = destino.with_suffix(".parquet.tmp")
    guardadas = 0
    inicio_archivo = 0
    try:
        with pq.ParquetWriter(destino_tmp, esquema_salida, compression="snappy") as escritor:
            for archivo, info in tqdm(list(zip(archivos, inventario, strict=True)), desc="Guardando train"):
                fin_archivo = inicio_archivo + info["filas"]
                seleccion = indices_train[np.searchsorted(indices_train, inicio_archivo):np.searchsorted(indices_train, fin_archivo)]
                if len(seleccion):
                    with pq.ParquetFile(archivo) as parquet:
                        inicio_grupo = inicio_archivo
                        for grupo in range(parquet.num_row_groups):
                            fin_grupo = inicio_grupo + parquet.metadata.row_group(grupo).num_rows
                            if np.searchsorted(seleccion, inicio_grupo) < np.searchsorted(seleccion, fin_grupo):
                                inicio_lote = inicio_grupo
                                for lote in parquet.iter_batches(batch_size=tamano_lote, row_groups=[grupo]):
                                    fin_lote = inicio_lote + lote.num_rows
                                    globales = seleccion[np.searchsorted(seleccion, inicio_lote):np.searchsorted(seleccion, fin_lote)]
                                    if len(globales):
                                        tabla = pa.Table.from_batches([lote]).take(pa.array(globales - inicio_lote))
                                        tabla = tabla.append_column("ind_cluster", pa.array(celdas[globales], type=pa.int32()))
                                        escritor.write_table(tabla)
                                        guardadas += len(globales)
                                    inicio_lote = fin_lote
                            inicio_grupo = fin_grupo
                inicio_archivo = fin_archivo
        if guardadas != len(indices_train):
            raise RuntimeError(f"Filas guardadas: {guardadas}; se esperaban {len(indices_train)}.")
        destino_tmp.replace(destino)
    finally:
        destino_tmp.unlink(missing_ok=True)
    return guardadas


guardadas = guardar_train(ruta_train, TAMANO_LOTE)

# Verificar lo escrito, no solo los índices seleccionados.
guardado = pq.read_table(ruta_train, columns=["image_id", "panoid", "ind_cluster"]).to_pandas()
for columna in ("image_id", "panoid"):
    valores = guardado[columna].astype("string").str.strip().replace("", pd.NA)
    if valores.isna().any() or valores.isin(validacion[columna]).any():
        raise RuntimeError(f"Train contiene {columna} inválidos o reservados para validación.")
if guardado["image_id"].duplicated().any() or (UNA_IMAGEN_POR_PANORAMA and guardado["panoid"].duplicated().any()):
    raise RuntimeError("Train contiene image_id o panoid duplicados.")
if not np.array_equal(np.bincount(guardado["ind_cluster"], minlength=N_CLUSTERS), seleccionadas.to_numpy()):
    raise RuntimeError("Las filas por celda guardadas no coinciden con los cupos.")
if pq.ParquetFile(ruta_train).schema_arrow.names != esquema.names + ["ind_cluster"]:
    raise RuntimeError("Las columnas guardadas no coinciden con la fuente más ind_cluster.")
comprobar_estado()

resumen.to_csv(ruta_resumen)
configuracion = {
    "tipo_muestra": "train_celdas_balanceado",
    "total_base": TOTAL,
    "total_pedido": TOTAL_OFICIAL,
    "filas_train": guardadas,
    "n_clusters": N_CLUSTERS,
    "mult_cont": MULT_CONT,
    "continente_manual": {int(k): v for k, v in CONTINENTE_MANUAL.items()},
    "cupo_min": int(cupos.min()),
    "cupo_max": int(cupos.max()),
    "celdas_con_deficit": int((deficit > 0).sum()),
    "deficit_total": int(deficit.sum()),
    "una_imagen_por_panorama": UNA_IMAGEN_POR_PANORAMA,
    "semilla": SEMILLA,
    "archivos_salida": {"train": ruta_train.name, "resumen": ruta_resumen.name},
    "prototipos": {"ruta": str(ruta_prototipos), "sha256": huella_prototipos},
    "validacion_global": info_validacion,
    "metodo": "exclusión de validación global por panoid e image_id + cupo por celda de Voronoi = MULT_CONT[continente] * TOTAL / N_CLUSTERS",
    "ruta_datos": str(ruta_datos),
    "patron_archivos": PATRON_ARCHIVOS,
    "archivos_fuente": inventario,
    "filas_fuente": len(celdas),
    "filas_ids_invalidos": int(ids_invalidos.sum()),
    "filas_coordenadas_invalidas": int(coords_invalidas.sum()),
    "filas_excluidas_validacion_global": int((reservadas & ~ids_invalidos & ~coords_invalidas).sum()),
}
ruta_config.write_text(json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Train guardado: {guardadas:,} imágenes en {ruta_train}")

Guardando train:   0%|          | 0/393 [00:00<?, ?it/s]

Train guardado: 28,204 imágenes en C:\Users\juanm\Documents\GitHub\geo-project\datasets\samples\data\train_aumentado_NA_EU\train_celdas_balanceado.parquet


## Validación con índice de celda

Escribe `validation_cluster.parquet` junto a `validation.parquet`: las mismas filas y columnas más
`ind_cluster` (int32), asignado con los mismos prototipos. `validation.parquet` no se modifica y la
salida no se sobrescribe si ya existe.

In [13]:
# Celda independiente: no depende de ninguna otra celda del notebook.
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# Parámetros editables.
ARCHIVO_PROTOTIPOS = "src/notebooks/preprocessing/assets/prototipos.csv"  # ind_cluster, lat, lon
VALIDACION_GLOBAL = "datasets/samples/data/validation/validation.parquet"
SALIDA_VALIDACION = "validation_cluster.parquet"  # Se crea junto a la validación.
COL_LAT, COL_LON = "latitude", "longitude"
TAMANO_LOTE = 128  # Limita la memoria al leer imágenes.

# Raíz del proyecto: primera carpeta (desde el directorio actual hacia arriba) que contiene los archivos.
raiz_proyecto = next(
    (c for c in (Path.cwd(), *Path.cwd().parents)
     if (c / ARCHIVO_PROTOTIPOS).is_file() and (c / VALIDACION_GLOBAL).is_file()),
    None,
)
if raiz_proyecto is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto con los prototipos y la validación.")


def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])


def asignar(X, C, chunk=200_000):
    # Celda del prototipo más cercano: mayor producto punto = menor distancia angular.
    labels = np.empty(len(X), dtype=np.int32)
    for i in range(0, len(X), chunk):
        labels[i:i + chunk] = (X[i:i + chunk] @ C.T).argmax(axis=1)
    return labels


def coordenadas_validas(lat, lon):
    return np.isfinite(lat) & np.isfinite(lon) & (np.abs(lat) <= 90) & (np.abs(lon) <= 180)


# Prototipos.
prototipos = pd.read_csv(raiz_proyecto / ARCHIVO_PROTOTIPOS)
if not {"ind_cluster", "lat", "lon"}.issubset(prototipos.columns):
    raise ValueError("El CSV de prototipos debe tener ind_cluster, lat y lon.")
prototipos = prototipos.sort_values("ind_cluster").reset_index(drop=True)
n_clusters = len(prototipos)
if n_clusters == 0 or not (prototipos["ind_cluster"].to_numpy() == np.arange(n_clusters)).all():
    raise ValueError("ind_cluster debe ser 0..N-1 sin repetidos ni huecos.")
if not coordenadas_validas(prototipos["lat"].to_numpy(float), prototipos["lon"].to_numpy(float)).all():
    raise ValueError("Hay prototipos con coordenadas inválidas.")
centroides_val = to_xyz(prototipos["lat"].to_numpy(float), prototipos["lon"].to_numpy(float))

# Validación con índice de celda.
ruta_val = raiz_proyecto / VALIDACION_GLOBAL
ruta_val_cluster = ruta_val.with_name(SALIDA_VALIDACION)
if ruta_val_cluster.exists():
    raise FileExistsError(f"La salida ya existe: {ruta_val_cluster}. No se sobrescribe.")

esquema_val = pq.ParquetFile(ruta_val).schema_arrow
if "ind_cluster" in esquema_val.names:
    raise ValueError("validation.parquet ya tiene una columna ind_cluster.")
esquema_val_salida = esquema_val.append(pa.field("ind_cluster", pa.int32()))
ruta_val_tmp = ruta_val_cluster.with_suffix(".parquet.tmp")
por_celda_val = np.zeros(n_clusters, dtype=np.int64)
filas_val = pq.ParquetFile(ruta_val).metadata.num_rows
escritas_val = 0
try:
    with pq.ParquetFile(ruta_val) as parquet, \
            pq.ParquetWriter(ruta_val_tmp, esquema_val_salida, compression="snappy") as escritor:
        for lote in tqdm(parquet.iter_batches(batch_size=TAMANO_LOTE), desc="Guardando validación"):
            lat = pd.to_numeric(lote.column(COL_LAT).to_pandas(), errors="coerce").to_numpy(float)
            lon = pd.to_numeric(lote.column(COL_LON).to_pandas(), errors="coerce").to_numpy(float)
            if not coordenadas_validas(lat, lon).all():
                raise ValueError("La validación global tiene coordenadas inválidas.")
            cluster = asignar(to_xyz(lat, lon), centroides_val)
            tabla = pa.Table.from_batches([lote]).append_column("ind_cluster", pa.array(cluster, type=pa.int32()))
            escritor.write_table(tabla)
            por_celda_val += np.bincount(cluster, minlength=n_clusters)
            escritas_val += len(cluster)
    if escritas_val != filas_val:
        raise RuntimeError(f"Filas guardadas: {escritas_val}; se esperaban {filas_val}.")
    ruta_val_tmp.replace(ruta_val_cluster)
finally:
    ruta_val_tmp.unlink(missing_ok=True)

print(f"Validación con cluster guardada: {escritas_val:,} imágenes en {ruta_val_cluster}")
print(f"Celdas sin imágenes de validación: {(por_celda_val == 0).sum()} de {n_clusters}")

FileExistsError: La salida ya existe: c:\Users\juanm\Documents\GitHub\geo-project\datasets\samples\data\validation\validation_cluster.parquet. No se sobrescribe.

## Mapa de las ubicaciones de train

Cada punto es una imagen de `train_celdas_balanceado.parquet` (se leen solo latitud, longitud y
`ind_cluster`), con un color distinto por celda. Las `X` negras son los prototipos.

In [14]:
import colorsys

import plotly.graph_objects as go

ubicaciones = pq.read_table(ruta_train, columns=[COL_LAT, COL_LON, "ind_cluster"]).to_pandas()


def color_celda(i):
    # Matiz por razón áurea para separar celdas vecinas; la luminosidad alterna ayuda a distinguirlas.
    r, g, b = colorsys.hls_to_rgb((i * 0.618033988749895) % 1.0, 0.45 if i % 2 else 0.6, 0.75)
    return f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})"


fig = go.Figure()
for celda, grupo in ubicaciones.groupby("ind_cluster", sort=True):
    fig.add_trace(go.Scattergeo(
        lat=grupo[COL_LAT], lon=grupo[COL_LON], mode="markers",
        marker=dict(size=3, color=color_celda(int(celda)), opacity=0.7),
        name=f"celda {celda}", text=f"celda {celda}", hoverinfo="text", showlegend=False,
    ))
fig.add_trace(go.Scattergeo(
    lat=prototipos["lat"], lon=prototipos["lon"], mode="markers+text",
    text=prototipos["ind_cluster"].astype(str), textposition="top center",
    marker=dict(size=8, color="black", symbol="x"), name="prototipos", showlegend=False,
))
fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f1ede4", showocean=True,
                oceancolor="#dfeaf2", showcountries=True, countrycolor="#bbbbbb", showcoastlines=True,
                coastlinecolor="#888888")
fig.update_layout(title=f"Train balanceado por celda: {len(ubicaciones):,} imágenes, {N_CLUSTERS} celdas",
                  height=650, margin=dict(l=0, r=0, t=50, b=0))
fig.show()